# API Financeira Integrada ao Banco Central do Brasil (BACEN)

Este notebook apresenta o desenvolvimento de uma API em Python
destinada à coleta, processamento e disponibilização de séries
temporais macroeconômicas provenientes do Banco Central do Brasil (BACEN).

A aplicação realiza:

- Consumo automatizado da API oficial do Sistema Gerenciador de Séries Temporais (SGS);
- Tratamento e validação estatística dos dados;
- Cálculos financeiros, como juros compostos e variação percentual acumulada;
- Disponibilização dos resultados por meio de endpoints REST.

O objetivo central é estruturar uma camada de serviço capaz de
fornecer dados confiáveis e preparados para análises temporais
no contexto do Motor Temporal desenvolvido neste trabalho.

## Cria dicionário códigos BACEN

In [ ]:
# Definição do conjunto de indicadores macroeconômicos
"""
Este dicionário define os indicadores macroeconômicos que serão utilizados
como variáveis de entrada no modelo de análise temporal.

A escolha dos indicadores segue critérios de relevância econômica
e impacto sistêmico sobre a dinâmica macroeconômica brasileira.

- SELIC: Representa a taxa básica de juros da economia brasileira.
  É o principal instrumento de política monetária utilizado pelo
  Banco Central para controle da inflação e regulação da atividade econômica.

- IPCA: Índice de Preços ao Consumidor Amplo.
  É o indicador oficial de inflação no Brasil, utilizado como
  referência para o regime de metas inflacionárias.

Os valores associados (11 para SELIC e 433 para IPCA) correspondem
aos códigos das séries temporais na API do Banco Central do Brasil (SGS),
permitindo a coleta automatizada dos dados históricos.

A estrutura em formato de dicionário permite:

1. Escalabilidade do modelo (inclusão de novos indicadores).
2. Iteração automatizada para coleta e processamento.
3. Organização semântica entre nome econômico e código técnico da série.

Este bloco é fundamental para o funcionamento do Motor Temporal,
pois define as variáveis que serão submetidas às análises de:
- Tendência linear
- Volatilidade móvel (rolling volatility)
- Correlação dinâmica
- Detecção de anomalias
"""

indicadores = {
    "SELIC": 11,  # Código da série da Taxa Selic no Banco Central (SGS)
    "IPCA": 433   # Código da série do IPCA no Banco Central (SGS)
}

## Função para buscar dados do BACEN

In [ ]:
# Função para Coleta de Dados Macroeconômicos via API do BACEN
def get_bacen_currency(serie_id: int, data_inicial=None, data_final=None):
    """
    Realiza a coleta automatizada de séries temporais macroeconômicas
    disponibilizadas pelo Banco Central do Brasil (BACEN), por meio da
    API oficial do Sistema Gerenciador de Séries Temporais (SGS).

    Parâmetros:
    ----------
    serie_id : int
        Código identificador da série no sistema SGS.
    data_inicial : str (opcional)
        Data inicial no formato DD/MM/AAAA.
    data_final : str (opcional)
        Data final no formato DD/MM/AAAA.

    Retorno:
    -------
    pandas.DataFrame
        DataFrame contendo:
        - coluna 'data' (datetime)
        - coluna 'valor' (float)
        ordenado cronologicamente.

    Fundamentação metodológica:
    ----------------------------
    A utilização da API oficial do BACEN garante:
    1. Confiabilidade e rastreabilidade da fonte de dados.
    2. Reprodutibilidade científica do experimento.
    3. Atualização automatizada das séries temporais.

    Este procedimento é essencial para assegurar integridade estatística
    antes da aplicação das análises do Motor Temporal.
    """

    # Construção dinâmica da URL da API SGS
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie_id}/dados?formato=json"

    # Caso o usuário defina intervalo temporal, adiciona filtros à requisição
    if data_inicial and data_final:
        url += f"&dataInicial={data_inicial}&dataFinal={data_final}"

    # Cabeçalho da requisição HTTP
    # O parâmetro "Accept: application/json" evita erro 406 (Not Acceptable)
    headers = {"Accept": "application/json"}

    # Requisição HTTP do tipo GET
    response = requests.get(url, headers=headers)

    # Validação da resposta da API
    # Garante robustez do pipeline contra falhas externas
    if response.status_code != 200:
        raise Exception(f"Erro ao acessar API do Bacen: {response.status_code}")

    # Conversão do JSON retornado para estrutura tabular
    data = response.json()
    df = pd.DataFrame(data)
    
    # Conversão da coluna de datas para formato datetime
    # dayfirst=True garante correta interpretação do padrão brasileiro (DD/MM/AAAA)
    df['data'] = pd.to_datetime(df['data'], dayfirst=True)

    # Conversão da coluna de valores para tipo numérico
    # errors='coerce' transforma inconsistências em NaN, evitando falhas posteriores
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')

    # Ordenação cronológica (importante para análises de séries temporais)
    df = df.sort_values('data').reset_index(drop=True)
    return df


## Função para Validar dados do BACEN

In [ ]:
# Função de Validação Estatística e Integridade da Série Temporal
def validar_dados(df):
    """
    Realiza verificação de integridade e consistência estatística
    em séries temporais macroeconômicas antes da aplicação de modelos.

    A validação contempla três dimensões fundamentais:

    1. Dados ausentes (missing values)
    2. Registros duplicados
    3. Detecção preliminar de outliers via Z-score

    Retorno:
    -------
    list
        Lista contendo alertas identificados na série.
        Caso não haja inconsistências, retorna lista vazia.

    Fundamentação metodológica:
    ----------------------------
    A validação prévia dos dados é etapa crítica em modelos de séries temporais,
    pois inconsistências podem comprometer:
    - Estimativas de tendência
    - Cálculo de volatilidade
    - Correlações móveis
    - Modelos ARIMA/SARIMA

    A detecção de outliers é realizada por meio do método de Z-score,
    que padroniza os dados considerando média e desvio padrão.
    Valores com |Z| > 3 são considerados estatisticamente extremos,
    assumindo aproximação à normalidade.
    """
    
    alertas = []

    # Verificação de valores ausentes
    # Dados faltantes podem distorcer médias, variâncias e modelos preditivos
    if df['valor'].isna().sum() > 0:
        alertas.append(f"{df['valor'].isna().sum()} valores ausentes")

     # Verificação de duplicidade temporal
     # Séries temporais devem possuir unicidade na dimensão temporal
    if df.duplicated(subset=['data']).sum() > 0:
        alertas.append(f"{df.duplicated(subset=['data']).sum()} duplicatas")
        
    #  Detecção de Outliers via Z-score
    df_mean = df['valor'].mean()
    df_std = df['valor'].std()

    # Evita divisão por zero (caso série seja constante)
    if df_std != 0:
         # Padronização estatística
        z_scores = (df['valor'] - df_mean) / df_std
        # Considera outliers valores com magnitude superior a 3 desvios padrão
        outliers = df[abs(z_scores) > 3]
        
        if not outliers.empty:
            alertas.append(f"{len(outliers)} outliers detectados: {outliers['valor'].tolist()}")
            
    return alertas

## Criar rota Flask 

In [ ]:
# Inicia o Flask
app = Flask(__name__)

@app.route('/')
def home():
    return jsonify({
        "mensagem": "API do Banco Central ativa",
        "endpoint_principal": "/indicadores",
        "exemplo": "/indicadores?inicio=01/01/2020&fim=01/01/2024"
    })

### Indicadores

In [ ]:
# Rota Flask para consulta de indicadores macroeconômicos
@app.route('/indicadores', methods=['GET'])
def consultar_indicadores():
    """
    Endpoint responsável por fornecer séries temporais macroeconômicas
    obtidas via API oficial do Banco Central do Brasil.

    Objetivo Arquitetural:
    ----------------------
    Esta rota atua como camada de serviço (Service Layer),
    disponibilizando dados tratados para consumo por módulos analíticos
    internos (MotorTemporal) ou aplicações externas.

    Parâmetros (Query Params):
    --------------------------
    inicio : str (formato DD/MM/AAAA)
        Data inicial da série temporal (default: 01/01/2016)
    fim : str (formato DD/MM/AAAA)
        Data final da série temporal (default: data atual)

    Retorno:
    --------
    JSON estruturado contendo:
    - ultimo_valor: valor mais recente da série
    - alertas: inconsistências identificadas na validação
    - historico: série temporal completa no período solicitado

    Justificativa Metodológica:
    ---------------------------
    A separação entre coleta, validação e análise garante:
    1. Modularização do sistema.
    2. Reprodutibilidade científica.
    3. Facilidade de auditoria e manutenção.
    """
    # Definição do intervalo temporal da consulta
    data_inicial = request.args.get("inicio", "01/01/2016")
    data_final = request.args.get(
        "fim",
        datetime.datetime.today().strftime("%d/%m/%Y")
    )

    # Estrutura que armazenará os resultados por indicador
    resultados = {}
     
    # Iteração sobre o dicionário de indicadores macroeconômicos
    for sigla, codigo in indicadores.items():
        try:

            # Tratamento específico para SELIC
            # A série da SELIC pode gerar erro 406 quando consultada
            # em períodos extensos. Para mitigar essa limitação da API,
            # a coleta é realizada em blocos anuais.
            if sigla == "SELIC":

                df_total = pd.DataFrame()
                ano_inicio = int(data_inicial.split("/")[2])
                ano_fim = int(data_final.split("/")[2])

                for ano in range(ano_inicio, ano_fim + 1):
                    inicio_ano = f"01/01/{ano}"
                    fim_ano = f"31/12/{ano}"

                    try:
                        df = get_bacen_currency(codigo, inicio_ano, fim_ano)

                        # Concatenação progressiva das séries anuais
                        df_total = pd.concat([df_total, df])
                        
                    except Exception as e:
                        # Log de erro não crítico (continua execução)
                        print(f"Ano {ano} apresentou erro: {e}")
                        
                # Reorganização cronológica após concatenação
                df = df_total.sort_values("data").reset_index(drop=True)

            else:
                # Coleta padrão para demais indicadores
                df = get_bacen_currency(codigo, data_inicial, data_final)

            # Verificação de disponibilidade de dados
            if df.empty:
                resultados[sigla] = {"erro": "Sem dados disponíveis"}
                continue

            # Validação de integridade estatística
            # Garante consistência antes do consumo analítico
            alertas = validar_dados(df)

            # Extração do valor mais recente
            # Indicador importante para monitoramento econômico
            ultimo_valor = df['valor'].iloc[-1]
            
            # Estrutura final retornada
            resultados[sigla] = {
                "ultimo_valor": ultimo_valor,
                "alertas": alertas,
                "historico": df.to_dict(orient="records")
            }

        except Exception as e:
            # Captura de falhas não previstas para evitar quebra total da API
            resultados[sigla] = {"erro": str(e)}
    # Serialização da resposta em formato JSON
    return jsonify(resultados)

In [ ]:
# roda Flask em thread para não travar o notebook
def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False)

t = Thread(target=run_app)
t.daemon = True
t.start()

print("API de indicadores rodando! Acesse http://127.0.0.1:5000/indicadores")


In [ ]:
print("API rodando em:")
print("http://127.0.0.1:5000/indicadores")

### Validando Flask

In [ ]:
# Testando e chamando a rota  Flask de indicador 
response = requests.get("http://127.0.0.1:5000/indicadores")
resultados = response.json()  # <- JSON retornado pela rota
print("Status code:", response.status_code)
